In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Tải dataset

* Bước 1: Tải dữ liệu từ [Google Drive](https://drive.google.com/drive/folders/1xclbjHHK58zk2X6iqbvMPS2rcy9y9E0X) của team UIT.
* Bước 2: Upload lên Google Colab theo cấu trúc
```
train/
  sents.txt
  sentiments.txt
test/
  sents.txt
  sentiments.txt
dev/
  sents.txt
  sentiments.txt
```

Lưu ý: tạo folder rồi up từng file txt lên sẽ dễ hơn.


In [ ]:
def load_vsfc(path):
    texts = []
    labels = []


    with open(os.path.join(path, "sents.txt"), encoding="utf-8") as f:
        texts = f.readlines()

    with open(os.path.join(path, "sentiments.txt"), encoding="utf-8") as f:
        labels = f.readlines()

    # strip newline
    texts = [t.strip() for t in texts]
    labels = [l.strip() for l in labels]

    return texts, labels

In [ ]:
train_texts, train_labels = load_vsfc("./train")
test_texts, test_labels = load_vsfc("./test")

print("Train size:", len(train_texts))
print("Test size:", len(test_texts))

Train size: 11426
Test size: 3166


In [ ]:
label_map = {"0": 0, "1": 1, "2": 2}

y_train = np.array([label_map[l] for l in train_labels])
y_test = np.array([label_map[l] for l in test_labels])

print("Sample labels:", y_train[:10])

Sample labels: [2 2 0 0 2 2 1 0 0 0]


In [ ]:
print(train_texts[:3])
print(train_labels[:3])

['slide giáo trình đầy đủ .', 'nhiệt tình giảng dạy , gần gũi với sinh viên .', 'đi học đầy đủ full điểm chuyên cần .']
['2', '2', '0']


In [ ]:
import re

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^0-9a-zA-Zà-ỹ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_texts = [preprocess(t) for t in train_texts]
test_texts = [preprocess(t) for t in test_texts]

print(train_texts[:3])

['slide giáo trình đầy đủ', 'nhiệt tình giảng dạy gần gũi với sinh viên', 'đi học đầy đủ full điểm chuyên cần']


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

In [ ]:
print("Vocab size thực tế:", len(tokenizer.word_index))

print("\nTop words:")
for word, idx in list(tokenizer.word_index.items())[:10]:
    print(word, "->", idx)

sample = train_texts[0]
print("\nSample:", sample)

seq = tokenizer.texts_to_sequences([sample])
print("Sequence:", seq)

print("\nWord mapping:")
for w in sample.split()[:5]:
    print(w, "->", tokenizer.word_index.get(w))

Vocab size thực tế: 2487

Top words:
<OOV> -> 1
viên -> 2
giảng -> 3
dạy -> 4
thầy -> 5
sinh -> 6
học -> 7
bài -> 8
tình -> 9
không -> 10

Sample: slide giáo trình đầy đủ
Sequence: [[120, 46, 52, 123, 91]]

Word mapping:
slide -> 120
giáo -> 46
trình -> 52
đầy -> 123
đủ -> 91


In [ ]:
X_train_seq = tokenizer.texts_to_sequences(train_texts)
X_test_seq = tokenizer.texts_to_sequences(test_texts)

In [ ]:

from tensorflow.keras.utils import pad_sequences

max_len = 50 # giải thích

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

print("Shape:", X_train_pad.shape)

Shape: (11426, 50)


In [ ]:
print("Before:", X_train_seq[0])
print("After :", X_train_pad[0])

Before: [120, 46, 52, 123, 91]
After : [120  46  52 123  91   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0]


In [ ]:
from tensorflow.keras import Input
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense


embedding_dim = 128


model = Sequential([
    Input(shape=(max_len,)),
    Embedding(vocab_size, embedding_dim),
    GlobalAveragePooling1D(),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 50, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,288,451 (4.92 MB)

 Trainable params: 1,288,451 (4.92 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.7585 - loss: 0.6081 - val_accuracy: 0.8679 - val_loss: 0.4104
Epoch 2/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.8774 - loss: 0.3668 - val_accuracy: 0.8425 - val_loss: 0.4246
Epoch 3/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.8901 - loss: 0.3272 - val_accuracy: 0.8696 - val_loss: 0.3616
Epoch 4/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9036 - loss: 0.2886 - val_accuracy: 0.8469 - val_loss: 0.4263
Epoch 5/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9074 - loss: 0.2722 - val_accuracy: 0.8705 - val_loss: 0.3588
Epoch 6/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.9140 - loss: 0.2540 - val_accuracy: 0.8819 - val_loss: 0.3295
Epoch 7/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9185 - loss: 0.2431 - val_accuracy: 0.8766 - val_loss: 0.3211
Epoch 8/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9224 - loss: 0.2299 - val_accu

In [ ]:
loss, acc = model.evaluate(X_test_pad, y_test)
print("Test Accuracy:", acc)

99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8816 - loss: 0.3641
Test Accuracy: 0.8815540075302124


In [ ]:
def predict_text(text):
    text = preprocess(text)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_len)
    pred = model.predict(pad)
    return np.argmax(pred)

print(predict_text("Giảng viên dạy rất hay"))
print(predict_text("Môn học này quá khó"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step
2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
0


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dropout, Dense

# 1. Định nghĩa kiến trúc mô hình BLSTM
embedding_dim = 128

model_blstm = Sequential([
    Input(shape=(max_len,)),
    Embedding(vocab_size, embedding_dim),
    Bidirectional(LSTM(64)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

In [ ]:
# 2. Compile mô hình
model_blstm.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_blstm.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 50, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,387,267 (5.29 MB)

 Trainable params: 1,387,267 (5.29 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 3. Huấn luyện mô hình
history_blstm = model_blstm.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 32s 85ms/step - accuracy: 0.8454 - loss: 0.4114 - val_accuracy: 0.8871 - val_loss: 0.3238
Epoch 2/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 25s 77ms/step - accuracy: 0.9156 - loss: 0.2520 - val_accuracy: 0.8994 - val_loss: 0.3077
Epoch 3/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 27s 83ms/step - accuracy: 0.9311 - loss: 0.2144 - val_accuracy: 0.9011 - val_loss: 0.2949
Epoch 4/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 26s 80ms/step - accuracy: 0.9420 - loss: 0.1842 - val_accuracy: 0.9029 - val_loss: 0.3171
Epoch 5/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 39s 75ms/step - accuracy: 0.9492 - loss: 0.1618 - val_accuracy: 0.8968 - val_loss: 0.3262
Epoch 6/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 41s 76ms/step - accuracy: 0.9540 - loss: 0.1452 - val_accuracy: 0.9116 - val_loss: 0.3014
Epoch 7/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 26s 82ms/step - accuracy: 0.9610 - loss: 0.1265 - val_accuracy: 0.9081 - val_loss: 0.3126
Epoch 8/10
322/322 ━━━━━━━━━━━━━━━━━━━━ 26s 82ms/step - accuracy: 0.9647 - loss: 0.1151 - 

In [ ]:
# 4. Đánh giá trên tập test
loss_blstm, acc_blstm = model_blstm.evaluate(X_test_pad, y_test)
print("\nTest Accuracy (BLSTM):", acc_blstm)

99/99 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.8986 - loss: 0.4332

Test Accuracy (BLSTM): 0.8986102342605591
